In [ ]:
%reload_ext autoreload
# %load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import itertools
import os
from g_anomaly import PA_with_anomaly_numba
from g_standard import standard_PA_numba
from support_tools import (
    edge_degree_counting,
    edge_degree_counting_many,
    expected_degree_PA_normal_all_vertices,
    graph_series_connected_vertex_degree,
    get_top_vertices,
    prepare_screening,
    get_vertex_ranks
)
from estimation_pa_anomaly_joint import iteration_estimation_update
from estimation_pa_anomaly_fixed import single_estimation_beta
from estimation_pa_normal import optimize_parameters_normal
from pa_loglikelihood import neg_log_likelihood_normal, neg_log_likelihood_anomaly

In [ ]:
def run_single_dataset_estimate_joint(beta_init, delta_init, network_size, m, edge_list, k_result, vertex_list):
    """
    top_k_degree: choose the vertices with top k degree as vertex candidates
    top_k_ratio: choose the vertices with top k ratio as vertex candidates
    vertex_i: 0-based, but in the calculation, vertex index should be 1-based
    log_likelihood_anomaly_numba: return the negative log-likelihood value
    procedure: 
        1. load the file, find the degree_k at every time step, and the vertices that might be anomaly
        2. estimation: find several candidates, multi-start of beta, minimize negative log-likelihood function
        3. find the vertex that has maximum log likelihood value
    return:
        tau_hat, beta_hat, delta_hat, log-likelihood value
    """
    
    beta_list, delta_list, log_L_list= [], [], []
        
    #edge_list = np.load(file_name).tolist()
    edges_np = np.asarray(edge_list, dtype=np.int64)          # shape (n,2)
    verts_np = np.asarray(vertex_list, dtype=np.int64)        # 0-based candidates

    deg_all, recv_all = edge_degree_counting_many(verts_np, edges_np, network_size)

    for i in range(len(vertex_list)):
        vertex_i = vertex_list[i] + 1  # likelihood: 1-based

 
        degree_series_i = deg_all[i].astype(np.float64)   
        edge_adj_i      = recv_all[i].astype(np.float64) 

        beta_i, delta_i, log_L_i = iteration_estimation_update(
            network_size, vertex_i, k_result,
            degree_series_i, edge_adj_i, m,
            beta_init, delta_init
        )
        log_L_list.append(log_L_i)
        beta_list.append(beta_i)
        delta_list.append(delta_i)
            #print(beta_i, delta_i, log_L_i)
    vertex_best = np.argmin(log_L_list)
    tau_hat = vertex_list[vertex_best]
    beta_hat = round(beta_list[vertex_best],4)
    delta_hat = round(delta_list[vertex_best],4)
    log_L_hat = round(log_L_list[vertex_best],4)

    return tau_hat, beta_hat, delta_hat, log_L_hat


In [ ]:
def main_detection_procedure_unknown_delta(base_seed, network_size, m, delta_true, beta_true, vertex_tau, Top_degree_rank, Top_ratio_rank, rep_num):
    
    """
    Our experiment is based on synthetic networks, the main detection procedure is:
    1. generate a PA network with an anomaly
    2. estimate delta_hat, beta_hat and tau_hat via likelihood maximization, and get the log likelihood ratio
    3. find the critical value:
       1) generate null network using delta_hat
       2) repeat step 2 to get the 95% quantile as the critical value
    4. compare the log likelihood ratio obtained from step 2 and the critical value, return the detection result
    """
    
    # initial values of beta and delta
    beta_init = 5.0
    delta_init = 0.001
    delta_r = 0   # this is for the value used in calculating the nominal ratio
    
    delta_bound_tmp = (-m, m)
    exp_degree_tmp = expected_degree_PA_normal_all_vertices(network_size, delta_true, m)

    results_list = []

    # --------- Under alternative (anomaly data) ---------
    for rep_1 in tqdm(range(rep_num), desc=f"Setting: tau = {vertex_tau}, beta = {beta_true}", leave=False):
        seed_1 = base_seed + rep_1

        edges_1 = PA_with_anomaly_numba(network_size, m, beta_true, vertex_tau, delta_true, seed_1)

        k_series_1 = np.asarray(graph_series_connected_vertex_degree(edges_1, receiver_index=1), dtype=np.float64)
        prepared = prepare_screening(edges_1, exp_degree_tmp, Top_degree_rank, Top_ratio_rank, delta_r, store_ratio=True)
        vertex_list_1 = prepared["vertex_list"]
        tau_hat, beta_hat, delta_hat, nll_1 = run_single_dataset_estimate_joint(
            beta_init, delta_init, network_size, m, edges_1, k_series_1, vertex_list_1)
        
        tau_d_rank, tau_r_rank = get_vertex_ranks(tau_hat, prepared,delta_true)

        delta0_1, nll0_1 = optimize_parameters_normal(delta_bound_tmp, network_size, k_series_1, m)

        log_lambda_1 = -nll_1 + nll0_1

        # --------- Plug-in delta_hat under null ---------
        log_lambda_0 = []
        for rep_2 in range(rep_num):
            seed_2 = base_seed + 500 + rep_2
            exp_degree_2 = expected_degree_PA_normal_all_vertices(network_size, delta_hat, m)

            edges_2 = standard_PA_numba(network_size, m, delta_hat, seed=seed_2)

            k_series_2 = np.asarray(graph_series_connected_vertex_degree(edges_2, receiver_index=1), dtype=np.float64)
            vertex_list_2 = get_top_vertices(edges_2, exp_degree_2, Top_degree_rank, Top_ratio_rank, delta_r)
            # The result from get_top_vertices is the same as prepare_screening, but the later will give the rank of degree and ratio
            #prepared_2 = prepare_screening(edges_2, exp_degree_2, Top_degree_rank, Top_ratio_rank, delta_r, store_ratio=True)
            #vertex_list_2 = prepared_2["vertex_list"]

            tau_hat_2, beta_hat_2, delta_hat_2, nll_2 = run_single_dataset_estimate_joint(
                beta_init, delta_init, network_size, m, edges_2, k_series_2, vertex_list_2)
            delta0_2, nll0_2 = optimize_parameters_normal(delta_bound_tmp, network_size, k_series_2, m)

            log_lambda_0.append(-nll_2 + nll0_2)

        c_0 = np.percentile(log_lambda_0, 95)
        if log_lambda_1 > c_0:
            detction_result = "Reject H_0"
        else:
            detction_result = "Accept H_0"
        rec = {
            "replicate": rep_1,
            "seed": seed_1,
            "beta_true": beta_true,
            "delta_true": delta_true,
            "tau_true": vertex_tau,
            "beta_hat": float(beta_hat),
            "delta_hat": float(delta_hat),
            "tau_hat": int(tau_hat),
            "degree_rank": tau_d_rank,
            "ratio_rank": tau_r_rank,
            "nll": float(nll_1),
            "critical_value": float(c_0),
            "nll0": float(nll0_1),
            "log_lambda": float(log_lambda_1),
            "detection_result": detction_result
        }
        
        results_list.append(rec)
    df_current = pd.DataFrame(results_list)
    return df_current

In [ ]:
def main():
    # sample parameters used for this run
    seed_num_tmp = 100
    network_size_tmp = 10000
    m_tmp = 3
    delta_true_tmp = 0.1
    beta_true_tmp = 0.01
    vertex_tau_tmp = 9700

    top_degree_k_tmp = 5
    top_ratio_k_tmp = 5
    rep_num_tmp = 100

    result_details = main_detection_procedure_unknown_delta(
        seed_num_tmp,
        network_size_tmp,
        m_tmp,
        delta_true_tmp,
        beta_true_tmp,
        vertex_tau_tmp,
        top_degree_k_tmp,
        top_ratio_k_tmp,
        rep_num_tmp
    )

    print(result_details)


if __name__ == "__main__":
    main()